<a href="https://colab.research.google.com/github/Sravani-939/genai-training-tasks/blob/main/webScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import json
from datetime import datetime


def fetch_html(url):
    """
    Tool 1: Download HTML from a webpage
    """
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return response.text


def extract_headlines(html):
    """
    Tool 2: Extract headlines from h1, h2, h3 tags
    """
    soup = BeautifulSoup(html, "html.parser")
    headlines = []

    for tag in soup.find_all(["h1", "h2", "h3"]):
        text = tag.get_text(strip=True)
        if text and len(text) > 15:
            headlines.append(text)

    # remove duplicates while keeping order
    unique_headlines = []
    seen = set()

    for h in headlines:
        if h not in seen:
            unique_headlines.append(h)
            seen.add(h)

    return unique_headlines[:10]


def extract_emails(text):
    """
    Tool 3: Extract email addresses using regex
    """
    pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
    emails = re.findall(pattern, text)

    # remove duplicate emails
    unique_emails = []
    seen = set()

    for e in emails:
        if e not in seen:
            unique_emails.append(e)
            seen.add(e)

    return unique_emails


def find_internal_links(base_url, html):
    """
    Tool 4: Find possible contact/about pages
    """
    soup = BeautifulSoup(html, "html.parser")
    links = []
    keywords = ["contact", "about", "editor", "team", "news", "media"]

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()

        if href.startswith("/"):
            full_url = base_url.rstrip("/") + href
        elif href.startswith("http"):
            full_url = href
        else:
            continue

        text = a.get_text(strip=True).lower()
        href_lower = href.lower()

        for key in keywords:
            if key in text or key in href_lower:
                links.append(full_url)
                break

    # removes duplicate links
    unique_links = []
    seen = set()

    for link in links:
        if link not in seen:
            unique_links.append(link)
            seen.add(link)

    return unique_links[:5]


# 2. AGENT CLASS
# This agent has:
# - memory
# - state
# - tool invocation
# - daily report saving

class SimpleNewsAgent:
    def __init__(self, website_list):
        self.website_list = website_list

        self.memory = {
            "visited_urls": [],
            "all_headlines_seen": [],
            "all_emails_seen": [],
            "logs": []
        }

        self.state = {
            "current_task": "idle",
            "current_url": "",
            "status": "not_started",
            "total_sites": len(website_list),
            "processed_sites": 0
        }

        self.results = []

    def remember(self, message):
        self.memory["logs"].append(message)


    def use_tool(self, tool_name, *args):
        self.remember(f"Using tool: {tool_name}")

        if tool_name == "fetch_html":
            return fetch_html(*args)

        elif tool_name == "extract_headlines":
            return extract_headlines(*args)

        elif tool_name == "extract_emails":
            return extract_emails(*args)

        elif tool_name == "find_internal_links":
            return find_internal_links(*args)

        else:
            raise ValueError(f"Unknown tool: {tool_name}")

    # Main agent work for one site
    def process_website(self, url):
        site_result = {
            "website": url,
            "main_page_headlines": [],
            "main_page_emails": [],
            "extra_page_emails": [],
            "status": "started",
            "error": ""
        }

        try:
            self.state["current_task"] = "scraping_main_page"
            self.state["current_url"] = url
            self.state["status"] = "working"
            self.remember(f"Started website: {url}")

            #fetch homepage
            html = self.use_tool("fetch_html", url)
            self.memory["visited_urls"].append(url)

            # extract headlines
            headlines = self.use_tool("extract_headlines", html)
            site_result["main_page_headlines"] = headlines

            # save headlines in memory
            for h in headlines:
                if h not in self.memory["all_headlines_seen"]:
                    self.memory["all_headlines_seen"].append(h)

            # extract emails from homepage
            emails = self.use_tool("extract_emails", html)
            site_result["main_page_emails"] = emails

            for e in emails:
                if e not in self.memory["all_emails_seen"]:
                    self.memory["all_emails_seen"].append(e)

            # if no emails found, check internal pages like contact/about
            self.state["current_task"] = "searching_internal_links"
            internal_links = self.use_tool("find_internal_links", url, html)

            extra_emails = []

            for link in internal_links:
                try:
                    self.remember(f"Checking internal page: {link}")
                    page_html = self.use_tool("fetch_html", link)
                    self.memory["visited_urls"].append(link)

                    page_emails = self.use_tool("extract_emails", page_html)

                    for e in page_emails:
                        if e not in extra_emails:
                            extra_emails.append(e)
                        if e not in self.memory["all_emails_seen"]:
                            self.memory["all_emails_seen"].append(e)

                except Exception as inner_error:
                    self.remember(f"Could not open internal page {link}: {str(inner_error)}")

            site_result["extra_page_emails"] = extra_emails
            site_result["status"] = "completed"

            self.remember(f"Finished website: {url}")

        except Exception as e:
            site_result["status"] = "failed"
            site_result["error"] = str(e)
            self.remember(f"Error on {url}: {str(e)}")

        self.results.append(site_result)
        self.state["processed_sites"] += 1


    def run(self):
        self.state["current_task"] = "starting"
        self.state["status"] = "running"
        self.remember("Agent started")

        for url in self.website_list:
            self.process_website(url)

        self.state["current_task"] = "finished"
        self.state["status"] = "completed"
        self.remember("Agent completed all websites")

    def make_daily_summary(self):
        today = datetime.now().strftime("%Y-%m-%d")
        lines = []
        lines.append("DAILY NEWS AND EMAIL UPDATE")
        lines.append(f"Date: {today}")
        lines.append("=" * 50)

        for item in self.results:
            lines.append(f"\nWebsite: {item['website']}")
            lines.append(f"Status: {item['status']}")

            if item["status"] == "completed":
                lines.append("Top Headlines:")
                if item["main_page_headlines"]:
                    for h in item["main_page_headlines"][:5]:
                        lines.append(f" - {h}")
                else:
                    lines.append(" - No headlines found")

                all_emails = item["main_page_emails"] + item["extra_page_emails"]
                unique_emails = []
                seen = set()

                for e in all_emails:
                    if e not in seen:
                        unique_emails.append(e)
                        seen.add(e)

                lines.append("Emails:")
                if unique_emails:
                    for e in unique_emails:
                        lines.append(f" - {e}")
                else:
                    lines.append(" - No emails found")

            else:
                lines.append(f"Error: {item['error']}")

        return "\n".join(lines)

    # Save output files
    def save_outputs(self):
        today = datetime.now().strftime("%Y-%m-%d")

        # Save CSV
        rows = []
        for item in self.results:
            all_emails = item["main_page_emails"] + item["extra_page_emails"]

            if item["main_page_headlines"]:
                for headline in item["main_page_headlines"]:
                    rows.append({
                        "date": today,
                        "website": item["website"],
                        "headline": headline,
                        "emails_found": ", ".join(all_emails),
                        "status": item["status"]
                    })
            else:
                rows.append({
                    "date": today,
                    "website": item["website"],
                    "headline": "",
                    "emails_found": ", ".join(all_emails),
                    "status": item["status"]
                })

        df = pd.DataFrame(rows)
        df.to_csv("daily_news_report.csv", index=False)

        # Saving memory and state in JSON
        save_json = {
            "memory": self.memory,
            "state": self.state,
            "results": self.results
        }

        with open("agent_memory_state.json", "w", encoding="utf-8") as f:
            json.dump(save_json, f, indent=4, ensure_ascii=False)

        # Save summary in TXT
        summary = self.make_daily_summary()
        with open("daily_summary.txt", "w", encoding="utf-8") as f:
            f.write(summary)

        print("Files saved:")
        print("1. daily_news_report.csv")
        print("2. agent_memory_state.json")
        print("3. daily_summary.txt")


#  using public pages which allow access

websites = [
    "https://www.bbc.com/news",
    "https://techcrunch.com",
    "https://www.theverge.com/tech"
]

agent = SimpleNewsAgent(websites)
agent.run()

print("\nFINAL STATE")
print(json.dumps(agent.state, indent=4))
# stores history
print("\nMEMORY LOGS")
for log in agent.memory["logs"]:
    print("-", log)

print("\nDAILY SUMMARY\n")
summary_text = agent.make_daily_summary()
print(summary_text)

agent.save_outputs()

# 7. SHOW CSV DATA
print("\nCSV Preview:")
df_preview = pd.read_csv("daily_news_report.csv")
display(df_preview.head(20))


FINAL STATE
{
    "current_task": "finished",
    "current_url": "https://www.theverge.com/tech",
    "status": "completed",
    "total_sites": 3,
    "processed_sites": 3
}

MEMORY LOGS
- Agent started
- Started website: https://www.bbc.com/news
- Using tool: fetch_html
- Using tool: extract_headlines
- Using tool: extract_emails
- Using tool: find_internal_links
- Checking internal page: https://www.bbc.com/news/watch-live-news/
- Using tool: fetch_html
- Could not open internal page https://www.bbc.com/news/watch-live-news/: 404 Client Error: Not Found for url: https://www.bbc.com/news/watch-live-news
- Checking internal page: https://www.bbc.com/news/news
- Using tool: fetch_html
- Could not open internal page https://www.bbc.com/news/news: 404 Client Error: Not Found for url: https://www.bbc.com/news/news
- Checking internal page: https://www.bbc.com/news/news/us-canada
- Using tool: fetch_html
- Could not open internal page https://www.bbc.com/news/news/us-canada: 404 Client Err

,date,website,headline,emails_found,status
0,2026-04-24,https://www.bbc.com/news,Trump says Israel-Lebanon ceasefire extended b...,NaN,completed
1,2026-04-24,https://www.bbc.com/news,US boards ship carrying Iran oil as Trump thre...,NaN,completed
2,2026-04-24,https://www.bbc.com/news,Ringo Starr tells BBC: 'I made all my mistakes...,NaN,completed
3,2026-04-24,https://www.bbc.com/news,Huge chunk of glacier blocks Everest route in ...,NaN,completed
4,2026-04-24,https://www.bbc.com/news,Masked Iranian forces appear to seize ships in...,NaN,completed
5,2026-04-24,https://www.bbc.com/news,Trump tells BBC that King's visit could 'absol...,NaN,completed
6,2026-04-24,https://www.bbc.com/news,"US soldier charged after winning $400,000 bett...",NaN,completed
7,2026-04-24,https://www.bbc.com/news,'Missing scientist' cases have stoked wild spe...,NaN,completed
8,2026-04-24,https://www.bbc.com/news,"Meta says it will cut 8,000 jobs as AI spendin...",NaN,completed
9,2026-04-24,https://www.bbc.com/news,Singer D4vd had 'significant amount' of child ...,NaN,completed
